In [ ]:
import wrds
import pandas as pd
import getpass

def main():
    # --- 1. Client-Facing Interface ---
    print("====================================================")
    print("📊 Welcome: CSMAR Fund Look-through Audit Tool")
    print("Description: This tool connects to the WRDS database to calculate ")
    print("the ROE of a fund's underlying stock holdings.")
    print("====================================================")
    
    # User Input Section
    wrds_user = input("👤 Enter your WRDS Username: ").strip()
    fund_code = input("🔢 Enter 6-digit Fund Code (e.g., 510300 or 512400): ").strip()
    report_date = input("📅 Enter Audit End Date (YYYY-MM-DD, e.g., 2022-12-31): ").strip()
    
    print("\n🚀 Establishing encrypted connection and starting audit procedure...\n")

    try:
        db = wrds.Connection(wrds_username=wrds_user)
        
        # 2. Retrieve Portfolio Holdings (symbol, proportion)
        print(f"Step 1/4: 🔍 Retrieving portfolio details for Fund {fund_code} as of {report_date}...")
        h_query = f"""
            SELECT symbol, proportion 
            FROM csmar_funds.fund_portfolio_stock 
            WHERE masterfundcode = '{fund_code}' AND enddate = '{report_date}'
        """
        holdings = db.raw_sql(h_query)
        
        if holdings.empty:
            print(f"❌ Error: No records found for Fund {fund_code} on the specified date.")
            return

        holdings['symbol'] = holdings['symbol'].astype(str).str.zfill(6)
        # Aggregate weights in case of duplicate entries
        holdings = holdings.groupby('symbol')['proportion'].sum().reset_index()

        # 3. Retrieve Financial Data (Using the verified 600k+ record production tables)
        print(f"Step 2/4: 📊 Auditing underlying Income Statements (fs_comins)...")
        i_query = f"""
            SELECT stkcd as symbol, b002100000 as net_income 
            FROM csmar_financial.fs_comins 
            WHERE accper = '{report_date}' AND typrep = 'A'
        """
        income = db.raw_sql(i_query)

        print(f"Step 3/4: 📊 Auditing underlying Balance Sheets (fs_combas)...")
        b_query = f"""
            SELECT stkcd as symbol, a001000000 as assets, a002000000 as liab 
            FROM csmar_financial.fs_combas 
            WHERE accper = '{report_date}' AND typrep = 'A'
        """
        balance = db.raw_sql(b_query)

        # 4. Name Mapping and Data Alignment
        print(f"Step 4/4: 🏷️ Mapping official stock names and calculating ROE metrics...")
        names = db.raw_sql("SELECT stkcd as symbol, stknme as stockname FROM csmar_trade.trd_co")

        # Data Cleaning
        income['symbol'] = income['symbol'].astype(str).str.zfill(6)
        balance['symbol'] = balance['symbol'].astype(str).str.zfill(6)
        names['symbol'] = names['symbol'].astype(str).str.zfill(6)

        # Core Merge and ROE Calculation Logic
        fin = pd.merge(income, balance, on='symbol', how='inner')
        fin['equity'] = fin['assets'] - fin['liab']
        # Filter for positive equity and remove duplicates
        fin = fin[fin['equity'] > 0].drop_duplicates('symbol')

        # Final Report Generation
        df = pd.merge(holdings, fin, on='symbol', how='left')
        df = pd.merge(df, names, on='symbol', how='left')
        df['ROE'] = df['net_income'] / df['equity']

        # 5. Output Audit Report
        top_15 = df.sort_values('proportion', ascending=False).head(15).copy()
        coverage = top_15['ROE'].notna().sum()
        
        print("\n" + "="*65)
        print(f"✅ Audit Report Successfully Generated!")
        print(f"📈 Top 15 Holdings Data Coverage: {coverage/15*100:.2f}%")
        print("-" * 65)
        print(top_15[['symbol', 'stockname', 'proportion', 'ROE']])
        print("-" * 65)
        
        output_name = f"Audit_Report_{fund_code}_{report_date.replace('-', '')}.xlsx"
        top_15.to_excel(output_name, index=False)
        print(f"📂 Audit working papers exported to Excel: {output_name}")
        print("====================================================\n")

    except Exception as e:
        print(f"❌ An error occurred during execution: {e}")
    finally:
        if 'db' in locals():
            db.close()

if __name__ == "__main__":
    main()

📊 Welcome: CSMAR Fund Look-through Audit Tool
Description: This tool connects to the WRDS database to calculate 
the ROE of a fund's underlying stock holdings.
